# Module 3: RAG Systems Engineering - Hybrid Search & GraphRAG

This notebook is refactored into smaller implementation blocks so each stage is easy to run and debug independently.


## 1. Setup: Dependencies and Runtime Configuration

First load all dependencies used throughout ingestion and retrieval.


In [1]:
import math
import json
import os
import sys
from typing import List, Dict, Any, Tuple, Optional
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict

import requests
import networkx as nx
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from dotenv import load_dotenv

# Add parent directory to path for logger_config
sys.path.append(str(Path.cwd().parent))
from logger_config import get_logger

# Set up logger
logger = get_logger(__name__)

print("✓ Imports successful")

✓ Imports successful


### Configuration Constants

Set service endpoints for OpenAI embeddings and Qdrant vector storage.


In [2]:
load_dotenv()
env = os.environ

# Ollama configuration
OLLAMA_BASE_URL = env.get("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_EMBED_MODEL = env.get("OLLAMA_EMBED_MODEL", "embeddinggemma:latest")
OLLAMA_MODEL = env.get("OLLAMA_MODEL", "gemma3:latest")

# Qdrant connection
QDRANT_URL = env.get("QDRANT_URL")

print("✓ Runtime configuration set")
print("✓ OLLAMA_BASE_URL loaded" if OLLAMA_BASE_URL else "⚠ OLLAMA_BASE_URL not found")
print("✓ OLLAMA_EMBED_MODEL loaded" if OLLAMA_EMBED_MODEL else "⚠ OLLAMA_EMBED_MODEL not found")
print("✓ QDRANT_URL loaded" if QDRANT_URL else "⚠ QDRANT_URL not found")

✓ Runtime configuration set
✓ OLLAMA_BASE_URL loaded
✓ OLLAMA_EMBED_MODEL loaded
✓ QDRANT_URL loaded


## 2. Document Ingestion with Hierarchical Chunking

Define the core chunk schema that preserves section structure and metadata.


In [3]:
@dataclass
class DocumentChunk:
    """Represents a chunk of text from the policy document."""

    chunk_id: str
    text: str
    section_number: str  # e.g., "4.1", "9.2.1"
    section_title: str   # e.g., "Taxis and Rideshares"
    parent_section: Optional[str]  # e.g., "4" for "4.1"
    page_number: int
    hierarchy_path: List[str]  # e.g., ["Section 4", "4.1"]
    chunk_type: str  # "section", "subsection", "paragraph"

    def to_dict(self) -> Dict[str, Any]:
        return {
            "chunk_id": self.chunk_id,
            "text": self.text,
            "section_number": self.section_number,
            "section_title": self.section_title,
            "parent_section": self.parent_section,
            "page_number": self.page_number,
            "hierarchy_path": self.hierarchy_path,
            "chunk_type": self.chunk_type,
        }

print("✓ DocumentChunk defined")

✓ DocumentChunk defined


### Ingestion Pipeline

The pipeline loads PDF pages, detects sections/subsections, and emits hierarchical chunks.


In [4]:
class DocumentIngestionPipeline:
    """Ingests PDF documents with hierarchical chunking."""

    def __init__(self, pdf_path: str):
        self.pdf_path = Path(pdf_path)
        self.chunks: List[DocumentChunk] = []
        logger.info(f"Initialized ingestion pipeline for: {self.pdf_path}")

    def load_pdf(self) -> List[Tuple[int, str]]:
        """Load PDF and extract text by page."""
        logger.info(f"Loading PDF: {self.pdf_path}")
        reader = PdfReader(self.pdf_path)
        pages = []

        for page_num, page in enumerate(reader.pages, start=1):
            text = page.extract_text()
            pages.append((page_num, text))

        logger.info(f"Loaded {len(pages)} pages from {self.pdf_path.name}")
        return pages

    def _create_section_chunk(self, section_num: str, section_title: str, page_num: int, chunk_counter: int) -> DocumentChunk:
        """Create a section-level chunk."""
        return DocumentChunk(
            chunk_id=f"chunk_{chunk_counter}",
            text=f"Section {section_num}: {section_title}",
            section_number=section_num,
            section_title=section_title,
            parent_section=None,
            page_number=page_num,
            hierarchy_path=[f"Section {section_num}"],
            chunk_type="section"
        )

    def _create_subsection_chunk(self, subsection_num: str, subsection_title: str, current_section: str, page_num: int, chunk_counter: int) -> DocumentChunk:
        """Create a subsection-level chunk."""
        return DocumentChunk(
            chunk_id=f"chunk_{chunk_counter}",
            text=f"{subsection_num} {subsection_title}",
            section_number=subsection_num,
            section_title=subsection_title,
            parent_section=current_section,
            page_number=page_num,
            hierarchy_path=[f"Section {current_section}", subsection_num],
            chunk_type="subsection"
        )

    def _create_subsubsection_chunk(self, subsubsection_num: str, subsubsection_title: str, current_subsection: str, page_num: int, chunk_counter: int) -> DocumentChunk:
        """Create a sub-subsection-level chunk."""
        return DocumentChunk(
            chunk_id=f"chunk_{chunk_counter}",
            text=f"{subsubsection_num} {subsubsection_title}",
            section_number=subsubsection_num,
            section_title=subsubsection_title,
            parent_section=current_subsection,
            page_number=page_num,
            hierarchy_path=[f"Section {current_subsection.split('.')[0]}", current_subsection, subsubsection_num],
            chunk_type="subsubsection"
        )

    def _create_paragraph_chunk(self, para: str, current_section: str, current_subsection: str, page_num: int, chunk_counter: int) -> DocumentChunk:
        """Create a paragraph-level chunk."""
        parent = current_subsection or current_section
        hierarchy = []
        if current_section:
            hierarchy.append(f"Section {current_section}")
        if current_subsection:
            hierarchy.append(current_subsection)

        return DocumentChunk(
            chunk_id=f"chunk_{chunk_counter}",
            text=para,
            section_number=parent or "0",
            section_title="",
            parent_section=parent,
            page_number=page_num,
            hierarchy_path=hierarchy,
            chunk_type="paragraph"
        )

    def _process_paragraph(self, para: str, current_section: str, current_subsection: str, page_num: int, chunk_counter: int, chunks: List[DocumentChunk]) -> Tuple[str, str, int]:
        """Process a single paragraph and update tracking variables."""
        import re

        # Patterns for section detection
        section_pattern = re.compile(r'^Section (\d+):\s*(.+)$', re.MULTILINE)
        subsection_pattern = re.compile(r'^(\d+\.\d+)\s+(.+)$', re.MULTILINE)
        subsubsection_pattern = re.compile(r'^(\d+\.\d+\.\d+)\s+(.+)$', re.MULTILINE)

        # Check for main section
        section_match = section_pattern.search(para)
        if section_match:
            section_num = section_match.group(1)
            section_title = section_match.group(2).strip()
            chunks.append(self._create_section_chunk(section_num, section_title, page_num, chunk_counter))
            return section_num, None, chunk_counter + 1

        # Check for subsection (e.g., 4.1)
        subsection_match = subsection_pattern.search(para)
        if subsection_match and current_section:
            subsection_num = subsection_match.group(1)
            subsection_title = subsection_match.group(2).strip()
            chunks.append(self._create_subsection_chunk(subsection_num, subsection_title, current_section, page_num, chunk_counter))
            return current_section, subsection_num, chunk_counter + 1

        # Check for sub-subsection (e.g., 4.1.1)
        subsubsection_match = subsubsection_pattern.search(para)
        if subsubsection_match and current_subsection:
            subsubsection_num = subsubsection_match.group(1)
            subsubsection_title = subsubsection_match.group(2).strip()
            chunks.append(self._create_subsubsection_chunk(subsubsection_num, subsubsection_title, current_subsection, page_num, chunk_counter))
            return current_section, current_subsection, chunk_counter + 1

        # Regular paragraph - attach to current subsection or section
        if len(para) > 50:  # Only chunk substantial paragraphs
            chunks.append(self._create_paragraph_chunk(para, current_section, current_subsection, page_num, chunk_counter))
            return current_section, current_subsection, chunk_counter + 1

        return current_section, current_subsection, chunk_counter

    def detect_sections(self, pages: List[Tuple[int, str]]) -> List[DocumentChunk]:
        """Detect hierarchical sections and create chunks."""
        logger.info("Starting section detection and chunking")
        chunks = []
        current_section = None
        current_subsection = None
        chunk_counter = 0

        for page_num, page_text in pages:
            # Skip title page and TOC (first 3 pages)
            if page_num <= 3:
                continue

            # Split into paragraphs
            paragraphs = [p.strip() for p in page_text.split('\n\n') if p.strip()]

            for para in paragraphs:
                current_section, current_subsection, chunk_counter = self._process_paragraph(
                    para, current_section, current_subsection, page_num, chunk_counter, chunks
                )

        logger.info(f"Created {len(chunks)} hierarchical chunks")
        return chunks

    def ingest(self) -> List[DocumentChunk]:
        """Run the full ingestion pipeline."""
        logger.info("Starting full ingestion pipeline")
        pages = self.load_pdf()
        self.chunks = self.detect_sections(pages)
        logger.info(f"Ingestion pipeline completed with {len(self.chunks)} chunks")
        return self.chunks

print("✓ DocumentIngestionPipeline defined")

✓ DocumentIngestionPipeline defined


### Run Ingestion

Ingest the sample policy and inspect a few chunk examples.


In [5]:
# Ingest the policy document
pipeline = DocumentIngestionPipeline("sample_expense_policy.pdf")
chunks = pipeline.ingest()

# Show some example chunks
print("\n=== Example Chunks ===")
for i, chunk in enumerate(chunks[:5]):
    print(f"\nChunk {i+1}:")
    print(f"  ID: {chunk.chunk_id}")
    print(f"  Section: {chunk.section_number} - {chunk.section_title}")
    print(f"  Parent: {chunk.parent_section}")
    print(f"  Page: {chunk.page_number}")
    print(f"  Hierarchy: {' > '.join(chunk.hierarchy_path)}")
    print(f"  Type: {chunk.chunk_type}")
    print(f"  Text preview: {chunk.text[:100]}...")

print(f"\n✓ Total chunks created: {len(chunks)}")

2026-03-01 13:07:52 - __main__ - INFO - Initialized ingestion pipeline for: sample_expense_policy.pdf
2026-03-01 13:07:52 - __main__ - INFO - Starting full ingestion pipeline
2026-03-01 13:07:52 - __main__ - INFO - Loading PDF: sample_expense_policy.pdf
incorrect startxref pointer(1)
parsing for Object Streams
2026-03-01 13:07:52 - __main__ - INFO - Loaded 17 pages from sample_expense_policy.pdf
2026-03-01 13:07:52 - __main__ - INFO - Starting section detection and chunking
2026-03-01 13:07:52 - __main__ - INFO - Created 14 hierarchical chunks
2026-03-01 13:07:52 - __main__ - INFO - Ingestion pipeline completed with 14 chunks



=== Example Chunks ===

Chunk 1:
  ID: chunk_0
  Section: 2 - Meals & Entertainment
  Parent: None
  Page: 4
  Hierarchy: Section 2
  Type: section
  Text preview: Section 2: Meals & Entertainment...

Chunk 2:
  ID: chunk_1
  Section: 2.2 - Client Dinners and Business Meals
  Parent: 2
  Page: 5
  Hierarchy: Section 2 > 2.2
  Type: subsection
  Text preview: 2.2 Client Dinners and Business Meals...

Chunk 3:
  ID: chunk_2
  Section: 2.3 - Alcohol Policy
  Parent: 2
  Page: 6
  Hierarchy: Section 2 > 2.3
  Type: subsection
  Text preview: 2.3 Alcohol Policy...

Chunk 4:
  ID: chunk_3
  Section: 3 - Lodging and Accommodations
  Parent: None
  Page: 7
  Hierarchy: Section 3
  Type: section
  Text preview: Section 3: Lodging and Accommodations...

Chunk 5:
  ID: chunk_4
  Section: 4 - Ground Transport
  Parent: None
  Page: 8
  Hierarchy: Section 4
  Type: section
  Text preview: Section 4: Ground Transport...

✓ Total chunks created: 14


# ## 3. Dense Vector Retrieval (Semantic Search)
# 
# Dense retrieval captures semantic similarity using Ollama embeddings and Qdrant.
# 

In [6]:
class DenseVectorRetriever:
    """Dense vector search using Ollama embeddings + Qdrant."""

    def __init__(self, collection_name: str = "policy_chunks"):
        self.collection_name = collection_name
        self.embed_model = OLLAMA_EMBED_MODEL
        self.embedding_dim = None  # Will be set after first embedding call
        self.client = QdrantClient(url=QDRANT_URL)
        logger.info(f"Initialized DenseVectorRetriever with collection: {collection_name}")

    def _get_embedding(self, text: str) -> List[float]:
        """Get Ollama embedding for a single text."""
        logger.debug(f"Getting embedding for text: {text[:50]}...")
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/embed",
            json={"model": self.embed_model, "input": text},
            timeout=30,
        )
        response.raise_for_status()
        embedding_data = response.json()
        
        # Handle different Ollama response formats
        if isinstance(embedding_data, dict) and "embeddings" in embedding_data:
            embedding = embedding_data["embeddings"][0]
        elif isinstance(embedding_data, list) and len(embedding_data) > 0:
            embedding = embedding_data[0]
        else:
            embedding = embedding_data
            
        # Set embedding dimension if not set yet
        if self.embedding_dim is None:
            self.embedding_dim = len(embedding)
            logger.info(f"Detected embedding dimension: {self.embedding_dim}")
            
        logger.debug(f"Embedding generated successfully (dim: {len(embedding)})")
        return embedding

    def _get_batch_embeddings(self, texts: List[str]) -> List[List[float]]:
        """Get Ollama embeddings for multiple texts (batch processing)."""
        logger.debug(f"Getting batch embeddings for {len(texts)} texts")
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/embed",
            json={"model": self.embed_model, "input": texts},
            timeout=60,
        )
        response.raise_for_status()
        embedding_data = response.json()
        
        # Handle different Ollama response formats
        if isinstance(embedding_data, dict) and "embeddings" in embedding_data:
            embeddings = embedding_data["embeddings"]
        elif isinstance(embedding_data, list):
            embeddings = embedding_data
        else:
            embeddings = [embedding_data]
            
        # Set embedding dimension if not set yet
        if self.embedding_dim is None and len(embeddings) > 0:
            self.embedding_dim = len(embeddings[0])
            logger.info(f"Detected embedding dimension: {self.embedding_dim}")
            
        logger.debug(f"Batch embeddings generated successfully (dim: {len(embeddings[0]) if embeddings else 0})")
        return embeddings

    def create_collection(self):
        """Create Qdrant collection for policy chunks."""
        if self.embedding_dim is None:
            # Get a test embedding to determine dimension
            logger.info("Getting test embedding to determine dimension...")
            test_embedding = self._get_embedding("test")
            self.embedding_dim = len(test_embedding)
            
        logger.info(f"Creating Qdrant collection: {self.collection_name} (dim: {self.embedding_dim})")
        try:
            self.client.delete_collection(self.collection_name)
        except Exception:
            # Collection might not exist, which is fine for creation
            pass

        self.client.create_collection(
            collection_name=self.collection_name,
            vectors_config=VectorParams(
                size=self.embedding_dim,
                distance=Distance.COSINE
            )
        )
        logger.info(f"Created Qdrant collection: {self.collection_name}")

    def index_chunks(self, chunks: List[DocumentChunk]):
        """Index document chunks in Qdrant using batch embeddings."""
        logger.info(f"Indexing {len(chunks)} chunks...")
        
        # Process in batches for efficiency
        batch_size = 10
        points = []
        
        for i in range(0, len(chunks), batch_size):
            batch_chunks = chunks[i:i + batch_size]
            batch_texts = [chunk.text for chunk in batch_chunks]
            
            logger.debug(f"Embedding batch {i//batch_size + 1}/{(len(chunks) + batch_size - 1)//batch_size}...")
            embeddings = self._get_batch_embeddings(batch_texts)
            
            for j, (chunk, embedding) in enumerate(zip(batch_chunks, embeddings)):
                point = PointStruct(
                    id=i + j,
                    vector=embedding,
                    payload=chunk.to_dict()
                )
                points.append(point)
        
        self.client.upsert(
            collection_name=self.collection_name,
            points=points
        )
        logger.info(f"Indexed {len(points)} chunks in Qdrant")

    def search(self, query: str, top_k: int = 5) -> List[Tuple[DocumentChunk, float]]:
        """Semantic search for relevant chunks."""
        logger.info(f"Performing semantic search for: {query[:50]}...")
        query_embedding = self._get_embedding(query)

        results = self.client.query_points(
            collection_name=self.collection_name,
            query=query_embedding,
            limit=top_k
        ).points

        retrieved = []
        for result in results:
            chunk = DocumentChunk(**result.payload)
            score = result.score
            retrieved.append((chunk, score))
        
        logger.info(f"Found {len(retrieved)} results for query")
        return retrieved

print("✓ DenseVectorRetriever defined")

✓ DenseVectorRetriever defined


# ## 3.1. Ollama Text Generation
# 
# Local text generation using Ollama for RAG synthesis and answer generation.
# 

In [10]:
def generate_answer_with_context(prompt: str, context_chunks: List[DocumentChunk], model: str = None) -> str:
    """Generate answer using Ollama with retrieved context."""
    if model is None:
        model = OLLAMA_MODEL
    
    logger.info(f"Generating answer using model: {model}")
    logger.debug(f"Context chunks: {len(context_chunks)}")
    
    # Handle both DocumentChunk objects and (DocumentChunk, score) tuples
    chunks = []
    for item in context_chunks:
        if isinstance(item, tuple) and len(item) == 2:
            chunk, score = item
            chunks.append(chunk)
        elif isinstance(item, DocumentChunk):
            chunks.append(item)
        else:
            logger.warning(f"Unexpected item type in context_chunks: {type(item)}")
    
    # Build context from retrieved chunks
    context_text = "\n\n".join([
        f"Source {i+1} (Section {chunk.section_number}): {chunk.text[:200]}..."
        for i, chunk in enumerate(chunks)
    ])
    
    # Build full prompt with context
    full_prompt = f"""Based on the following policy document excerpts, answer the question concisely:

Context:
{context_text}

Question: {prompt}

Answer:"""

    try:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json={
                "model": model,
                "prompt": full_prompt,
                "max_tokens": 512,
                "temperature": 0.1,
                "stream": False
            },
            timeout=60
        )
        response.raise_for_status()
        
        result = response.json()
        answer = result.get("response", "")
        
        logger.info(f"Generated answer (length: {len(answer)})")
        return answer
        
    except Exception as e:
        logger.error(f"Error generating answer: {e}")
        return f"Error generating answer: {str(e)}"

print("✓ Ollama generation function defined")

✓ Ollama generation function defined


### Build Dense Index

Create the vector collection and upload chunk embeddings.


In [11]:
# Create dense retriever and index chunks
dense_retriever = DenseVectorRetriever()
dense_retriever.create_collection()
dense_retriever.index_chunks(chunks)

print("\n✓ Dense vector indexing complete")

2026-03-01 13:10:02 - __main__ - INFO - Initialized DenseVectorRetriever with collection: policy_chunks
2026-03-01 13:10:02 - __main__ - INFO - Getting test embedding to determine dimension...
2026-03-01 13:10:02 - __main__ - INFO - Detected embedding dimension: 768
2026-03-01 13:10:02 - __main__ - INFO - Creating Qdrant collection: policy_chunks (dim: 768)
2026-03-01 13:10:06 - __main__ - INFO - Created Qdrant collection: policy_chunks
2026-03-01 13:10:06 - __main__ - INFO - Indexing 14 chunks...
2026-03-01 13:10:09 - __main__ - INFO - Indexed 14 chunks in Qdrant



✓ Dense vector indexing complete


# ### Test Ollama Generation
# 
# Test the local text generation with retrieved context.
# 

In [12]:
# Test Ollama generation with retrieved context
test_query = "What is the meal limit for employees?"
retrieved_chunks = dense_retriever.search(test_query, top_k=3)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved_chunks)} chunks")

# Generate answer using Ollama
if retrieved_chunks:
    answer = generate_answer_with_context(test_query, retrieved_chunks)
    print("\nGenerated Answer:")
    print(answer)
else:
    print("No chunks retrieved for generation test")

2026-03-01 13:10:09 - __main__ - INFO - Performing semantic search for: What is the meal limit for employees?...
2026-03-01 13:10:10 - __main__ - INFO - Found 3 results for query
2026-03-01 13:10:10 - __main__ - INFO - Generating answer using model: gemma3:latest


Query: What is the meal limit for employees?
Retrieved 3 chunks


2026-03-01 13:11:00 - __main__ - INFO - Generated answer (length: 178)



Generated Answer:
The policy excerpts don't specify a meal limit for employees. They primarily address client dinners, business meals for VPs and above, and general meals & entertainment policies.


### Test Dense Search

Run semantic queries and inspect top results.


In [13]:
# Test semantic search
test_queries = [
    "What is the meal limit for individual employees?",
    "Can I expense sustenance while traveling?",
    "What are the rules for client dinners?",
]

print("=== Dense Vector Search Results ===")
for query in test_queries:
    print(f"\nQuery: {query}")
    results = dense_retriever.search(query, top_k=3)

    for i, (chunk, score) in enumerate(results, 1):
        print(f"\n  Result {i} (score: {score:.3f}):")
        print(f"    Section: {chunk.section_number} - {chunk.section_title}")
        print(f"    Page: {chunk.page_number}")
        print(f"    Text: {chunk.text[:150]}...")

2026-03-01 13:11:00 - __main__ - INFO - Performing semantic search for: What is the meal limit for individual employees?...


=== Dense Vector Search Results ===

Query: What is the meal limit for individual employees?


2026-03-01 13:11:09 - __main__ - INFO - Found 3 results for query
2026-03-01 13:11:09 - __main__ - INFO - Performing semantic search for: Can I expense sustenance while traveling?...



  Result 1 (score: 0.479):
    Section: 2.2 - Client Dinners and Business Meals
    Page: 5
    Text: 2.2 Client Dinners and Business Meals...

  Result 2 (score: 0.442):
    Section: 9.2 - VP and Above - Meal and Entertainment
    Page: 16
    Text: 9.2 VP and Above - Meal and Entertainment...

  Result 3 (score: 0.413):
    Section: 2 - Meals & Entertainment
    Page: 4
    Text: Section 2: Meals & Entertainment...

Query: Can I expense sustenance while traveling?


2026-03-01 13:11:10 - __main__ - INFO - Found 3 results for query
2026-03-01 13:11:10 - __main__ - INFO - Performing semantic search for: What are the rules for client dinners?...



  Result 1 (score: 0.522):
    Section: 8 - Miscellaneous Expenses
    Page: 14
    Text: Section 8: Miscellaneous Expenses...

  Result 2 (score: 0.488):
    Section: 6 - International Travel
    Page: 12
    Text: Section 6: International Travel...

  Result 3 (score: 0.445):
    Section: 2 - Meals & Entertainment
    Page: 4
    Text: Section 2: Meals & Entertainment...

Query: What are the rules for client dinners?


2026-03-01 13:11:11 - __main__ - INFO - Found 3 results for query



  Result 1 (score: 0.633):
    Section: 2.2 - Client Dinners and Business Meals
    Page: 5
    Text: 2.2 Client Dinners and Business Meals...

  Result 2 (score: 0.438):
    Section: 2 - Meals & Entertainment
    Page: 4
    Text: Section 2: Meals & Entertainment...

  Result 3 (score: 0.413):
    Section: 9.2 - VP and Above - Meal and Entertainment
    Page: 16
    Text: 9.2 VP and Above - Meal and Entertainment...


## 4. Sparse BM25 Retrieval (Keyword Search)

Sparse retrieval is effective for exact codes, amounts, and phrase matching.


In [14]:
class SparseBM25Retriever:
    """Sparse keyword retrieval using BM25."""

    def __init__(self):
        self.chunks: List[DocumentChunk] = []
        self.bm25: Optional[BM25Okapi] = None
        self.tokenized_corpus: List[List[str]] = []

    def _tokenize(self, text: str) -> List[str]:
        """Simple tokenization: lowercase and split on whitespace."""
        return text.lower().split()

    def index_chunks(self, chunks: List[DocumentChunk]):
        """Build BM25 index over chunks."""
        self.chunks = chunks
        self.tokenized_corpus = [self._tokenize(chunk.text) for chunk in chunks]
        self.bm25 = BM25Okapi(self.tokenized_corpus)
        print(f"✓ Built BM25 index over {len(chunks)} chunks")

    def search(self, query: str, top_k: int = 5) -> List[Tuple[DocumentChunk, float]]:
        """Keyword search for relevant chunks."""
        if not self.bm25:
            raise ValueError("BM25 index not built. Call index_chunks() first.")

        tokenized_query = self._tokenize(query)
        scores = self.bm25.get_scores(tokenized_query)
        top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

        results = [(self.chunks[i], scores[i]) for i in top_indices]
        return results

print("✓ SparseBM25Retriever defined")

✓ SparseBM25Retriever defined


### Build and Test BM25 Index

Run keyword-heavy queries including policy codes and dollar limits.


In [15]:
# Create BM25 retriever and index chunks
bm25_retriever = SparseBM25Retriever()
bm25_retriever.index_chunks(chunks)

# Test keyword search
test_queries = [
    "PROJ-2024-001",
    "$150 client dinner",
    "VP first class",
]

print("\n=== BM25 Keyword Search Results ===")
for query in test_queries:
    print(f"\nQuery: {query}")
    results = bm25_retriever.search(query, top_k=3)

    for i, (chunk, score) in enumerate(results, 1):
        print(f"\n  Result {i} (score: {score:.3f}):")
        print(f"    Section: {chunk.section_number} - {chunk.section_title}")
        print(f"    Page: {chunk.page_number}")
        print(f"    Text: {chunk.text[:150]}...")

✓ Built BM25 index over 14 chunks

=== BM25 Keyword Search Results ===

Query: PROJ-2024-001

  Result 1 (score: 0.000):
    Section: 2 - Meals & Entertainment
    Page: 4
    Text: Section 2: Meals & Entertainment...

  Result 2 (score: 0.000):
    Section: 2.2 - Client Dinners and Business Meals
    Page: 5
    Text: 2.2 Client Dinners and Business Meals...

  Result 3 (score: 0.000):
    Section: 2.3 - Alcohol Policy
    Page: 6
    Text: 2.3 Alcohol Policy...

Query: $150 client dinner

  Result 1 (score: 1.926):
    Section: 2.2 - Client Dinners and Business Meals
    Page: 5
    Text: 2.2 Client Dinners and Business Meals...

  Result 2 (score: 0.000):
    Section: 2 - Meals & Entertainment
    Page: 4
    Text: Section 2: Meals & Entertainment...

  Result 3 (score: 0.000):
    Section: 2.3 - Alcohol Policy
    Page: 6
    Text: 2.3 Alcohol Policy...

Query: VP first class

  Result 1 (score: 1.643):
    Section: 9.2 - VP and Above - Meal and Entertainment
    Page: 16
    Text:

## 5. Hybrid Search: Dense + Sparse Fusion

Hybrid retrieval combines semantic recall and exact-match precision using Reciprocal Rank Fusion (RRF).


In [16]:
class HybridRetriever:
    """Hybrid retrieval combining dense vector and sparse BM25 search."""

    def __init__(
        self,
        dense_retriever: DenseVectorRetriever,
        sparse_retriever: SparseBM25Retriever,
        dense_weight: float = 0.5,
        sparse_weight: float = 0.5,
    ):
        self.dense_retriever = dense_retriever
        self.sparse_retriever = sparse_retriever
        self.dense_weight = dense_weight
        self.sparse_weight = sparse_weight

    def reciprocal_rank_fusion(
        self,
        dense_results: List[Tuple[DocumentChunk, float]],
        sparse_results: List[Tuple[DocumentChunk, float]],
        k: int = 60,
    ) -> List[Tuple[DocumentChunk, float]]:
        """Merge results using Reciprocal Rank Fusion.

        RRF score = sum(1 / (k + rank)) for each retriever where doc appears.
        """
        scores = defaultdict(float)
        chunk_map = {}

        for rank, (chunk, _) in enumerate(dense_results, start=1):
            rrf_score = self.dense_weight / (k + rank)
            scores[chunk.chunk_id] += rrf_score
            chunk_map[chunk.chunk_id] = chunk

        for rank, (chunk, _) in enumerate(sparse_results, start=1):
            rrf_score = self.sparse_weight / (k + rank)
            scores[chunk.chunk_id] += rrf_score
            chunk_map[chunk.chunk_id] = chunk

        sorted_chunks = sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )

        return [(chunk_map[chunk_id], score) for chunk_id, score in sorted_chunks]

    def search(self, query: str, top_k: int = 5) -> List[Tuple[DocumentChunk, float]]:
        """Hybrid search combining dense and sparse retrieval."""
        dense_results = self.dense_retriever.search(query, top_k=top_k * 2)
        sparse_results = self.sparse_retriever.search(query, top_k=top_k * 2)
        hybrid_results = self.reciprocal_rank_fusion(dense_results, sparse_results)
        return hybrid_results[:top_k]

print("✓ HybridRetriever defined")

✓ HybridRetriever defined


### Test Hybrid Search

Run mixed semantic + keyword queries and inspect fused ranking.


In [17]:
# Create hybrid retriever
hybrid_retriever = HybridRetriever(
    dense_retriever=dense_retriever,
    sparse_retriever=bm25_retriever,
    dense_weight=0.6,  # Slightly favor semantic search
    sparse_weight=0.4,
)

# Test hybrid search
test_queries = [
    "What is the meal allowance for sustenance?",  # Semantic query
    "PROJ-2024-002 client development",  # Keyword query
    "Can VP fly first class?",  # Mixed query
]

print("=== Hybrid Search Results ===")
for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")

    results = hybrid_retriever.search(query, top_k=3)

    for i, (chunk, score) in enumerate(results, 1):
        print(f"\nResult {i} (RRF score: {score:.4f}):")
        print(f"  Section: {chunk.section_number} - {chunk.section_title}")
        print(f"  Page: {chunk.page_number}")
        print(f"  Hierarchy: {' > '.join(chunk.hierarchy_path)}")
        print(f"  Text: {chunk.text[:200]}...")

2026-03-01 13:11:11 - __main__ - INFO - Performing semantic search for: What is the meal allowance for sustenance?...


=== Hybrid Search Results ===

Query: What is the meal allowance for sustenance?


2026-03-01 13:11:12 - __main__ - INFO - Found 6 results for query
2026-03-01 13:11:12 - __main__ - INFO - Performing semantic search for: PROJ-2024-002 client development...



Result 1 (RRF score: 0.0164):
  Section: 9.2 - VP and Above - Meal and Entertainment
  Page: 16
  Hierarchy: Section 9 > 9.2
  Text: 9.2 VP and Above - Meal and Entertainment...

Result 2 (RRF score: 0.0161):
  Section: 2 - Meals & Entertainment
  Page: 4
  Hierarchy: Section 2
  Text: Section 2: Meals & Entertainment...

Result 3 (RRF score: 0.0159):
  Section: 2.2 - Client Dinners and Business Meals
  Page: 5
  Hierarchy: Section 2 > 2.2
  Text: 2.2 Client Dinners and Business Meals...

Query: PROJ-2024-002 client development


2026-03-01 13:11:12 - __main__ - INFO - Found 6 results for query
2026-03-01 13:11:12 - __main__ - INFO - Performing semantic search for: Can VP fly first class?...



Result 1 (RRF score: 0.0164):
  Section: 2.2 - Client Dinners and Business Meals
  Page: 5
  Hierarchy: Section 2 > 2.2
  Text: 2.2 Client Dinners and Business Meals...

Result 2 (RRF score: 0.0157):
  Section: 4.2 - Rental Cars
  Page: 9
  Hierarchy: Section 4 > 4.2
  Text: 4.2 Rental Cars...

Result 3 (RRF score: 0.0095):
  Section: 7 - Conference and Training
  Page: 13
  Hierarchy: Section 7
  Text: Section 7: Conference and Training...

Query: Can VP fly first class?


2026-03-01 13:11:13 - __main__ - INFO - Found 6 results for query



Result 1 (RRF score: 0.0161):
  Section: 9.2 - VP and Above - Meal and Entertainment
  Page: 16
  Hierarchy: Section 9 > 9.2
  Text: 9.2 VP and Above - Meal and Entertainment...

Result 2 (RRF score: 0.0152):
  Section: 4 - Ground Transport
  Page: 8
  Hierarchy: Section 4
  Text: Section 4: Ground Transport...

Result 3 (RRF score: 0.0098):
  Section: 5 - Air Travel
  Page: 11
  Hierarchy: Section 5
  Text: Section 5: Air Travel...


# ## 6. Complete RAG Pipeline with Ollama
# 
# Demonstrate end-to-end RAG using Ollama embeddings and generation.
# 

In [18]:
def rag_pipeline(query: str, retriever: HybridRetriever, model: str = None) -> Dict[str, Any]:
    """Complete RAG pipeline using Ollama."""
    logger.info(f"Running RAG pipeline for query: {query}")
    
    # Retrieve relevant chunks
    retrieved_chunks = retriever.search(query, top_k=3)
    
    if not retrieved_chunks:
        return {
            "query": query,
            "answer": "I couldn't find relevant information in the policy document.",
            "sources": []
        }
    
    # Generate answer using Ollama
    answer = generate_answer_with_context(query, retrieved_chunks, model)
    
    # Prepare source information
    sources = [
        {
            "section": chunk.section_number,
            "title": chunk.section_title,
            "page": chunk.page_number,
            "text_preview": chunk.text[:100] + "...",
            "score": score
        }
        for chunk, score in retrieved_chunks
    ]
    
    return {
        "query": query,
        "answer": answer,
        "sources": sources
    }

# Test complete RAG pipeline
print("=== Complete RAG Pipeline Test ===")

test_questions = [
    "What is the meal allowance for employees?",
    "Can I expense client entertainment?",
    "What are the travel expense rules?",
]

for question in test_questions:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}")
    
    result = rag_pipeline(question, hybrid_retriever)
    
    print("\nAnswer:")
    print(result["answer"])
    
    print("\nSources:")
    for i, source in enumerate(result["sources"], 1):
        print(f"  {i}. Section {source['section']} - {source['title']} (Page {source['page']})")
        print(f"     Score: {source['score']:.3f}")
        print(f"     Preview: {source['text_preview']}")

2026-03-01 13:11:13 - __main__ - INFO - Running RAG pipeline for query: What is the meal allowance for employees?
2026-03-01 13:11:13 - __main__ - INFO - Performing semantic search for: What is the meal allowance for employees?...


=== Complete RAG Pipeline Test ===

Question: What is the meal allowance for employees?


2026-03-01 13:11:14 - __main__ - INFO - Found 6 results for query
2026-03-01 13:11:14 - __main__ - INFO - Generating answer using model: gemma3:latest
2026-03-01 13:11:49 - __main__ - INFO - Generated answer (length: 211)
2026-03-01 13:11:49 - __main__ - INFO - Running RAG pipeline for query: Can I expense client entertainment?
2026-03-01 13:11:49 - __main__ - INFO - Performing semantic search for: Can I expense client entertainment?...



Answer:
The policy document doesn't specify a particular meal allowance. It references provisions for “Meal and Entertainment” for VP and above (Source 1) and outlines “Client Dinners and Business Meals” (Source 2 & 3).

Sources:
  1. Section 9.2 - VP and Above - Meal and Entertainment (Page 16)
     Score: 0.016
     Preview: 9.2 VP and Above - Meal and Entertainment...
  2. Section 2.2 - Client Dinners and Business Meals (Page 5)
     Score: 0.016
     Preview: 2.2 Client Dinners and Business Meals...
  3. Section 2 - Meals & Entertainment (Page 4)
     Score: 0.016
     Preview: Section 2: Meals & Entertainment...

Question: Can I expense client entertainment?


2026-03-01 13:11:50 - __main__ - INFO - Found 6 results for query
2026-03-01 13:11:50 - __main__ - INFO - Generating answer using model: gemma3:latest
2026-03-01 13:12:12 - __main__ - INFO - Generated answer (length: 144)
2026-03-01 13:12:12 - __main__ - INFO - Running RAG pipeline for query: What are the travel expense rules?
2026-03-01 13:12:12 - __main__ - INFO - Performing semantic search for: What are the travel expense rules?...



Answer:
Yes, according to these excerpts (specifically Section 2.2 and 2), client entertainment (meals & business meals) is addressed within the policy.

Sources:
  1. Section 2.2 - Client Dinners and Business Meals (Page 5)
     Score: 0.016
     Preview: 2.2 Client Dinners and Business Meals...
  2. Section 2 - Meals & Entertainment (Page 4)
     Score: 0.016
     Preview: Section 2: Meals & Entertainment...
  3. Section 3 - Lodging and Accommodations (Page 7)
     Score: 0.015
     Preview: Section 3: Lodging and Accommodations...

Question: What are the travel expense rules?


2026-03-01 13:12:15 - __main__ - INFO - Found 6 results for query
2026-03-01 13:12:15 - __main__ - INFO - Generating answer using model: gemma3:latest
2026-03-01 13:12:34 - __main__ - INFO - Generated answer (length: 128)



Answer:
The travel expense rules are outlined in Sections 5 (Air Travel), 6 (International Travel), and 3 (Lodging and Accommodations).


Sources:
  1. Section 5 - Air Travel (Page 11)
     Score: 0.016
     Preview: Section 5: Air Travel...
  2. Section 6 - International Travel (Page 12)
     Score: 0.016
     Preview: Section 6: International Travel...
  3. Section 3 - Lodging and Accommodations (Page 7)
     Score: 0.015
     Preview: Section 3: Lodging and Accommodations...
